## Day 1

Created main window with File/Help menus and action items

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout, \
    QLabel, QWidget, QGridLayout, QLineEdit, QPushButton,QMainWindow  # Import Qt widgets for GUI
from PyQt6.QtGui import QAction  # Import QAction for menu items
import sys  # Import sys for system-related operations (argv, exit)

class MainWindow(QMainWindow):
    """Main application window for Student Management System"""

    def __init__(self):
        """Initialize the main window and set up menu bar"""
        super().__init__()  # Call parent class constructor
        self.setWindowTitle("Student Management System")  # Set window title

        # Create File and Help menus on the menu bar
        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")

        # Create "Add Student" action and add it to File menu
        add_student_action = QAction("Add Student", self)
        file_menu_item.addAction(add_student_action)

        # Create "About" action and add it to Help menu
        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        # about_action.setMenuRole(QAction.MenuRole.NoRole)  # (Commented) Would prevent About from moving to OS menu

# Create QApplication instance (required for any Qt GUI)
app = QApplication(sys.argv)
# Create and show the main window
main_window = MainWindow()
main_window.show()
# Start the event loop and exit when closed
sys.exit(app.exec())

Integrated student data table (4 columns) into main window

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout,QTableWidget, \
    QLabel, QWidget, QGridLayout, QLineEdit, QPushButton,QMainWindow  # NEW: Added QTableWidget
from PyQt6.QtGui import QAction 
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""

    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")


        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")

        add_student_action = QAction("Add Student", self)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)

        #Table widget setup
        self.table = QTableWidget()  # Create table instance
        self.table.setColumnCount(4)  # Define 4 columns
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Coures", "Mobile"))  # Set column headers
        self.setCentralWidget(self.table)  # Display table as main content

    def load_data(self):
        """Placeholder method for loading student data into table (not implemented yet)"""
        self.table


app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
sys.exit(app.exec())

## Day 2

Implemented student insertion dialog with SQLite database connection and course combobox

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout,QTableWidget, \
    QLabel, QWidget, QGridLayout, QLineEdit, QPushButton,QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox  # Added QTableWidgetItem, QDialog, QComboBox
from PyQt6.QtGui import QAction
import sqlite3  # For database operations
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")

        add_student_action = QAction("Add Student", self)
        add_student_action.triggered.connect(self.insert)  # Connect to insert method
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)  # Prevent About from moving to OS menu

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Coures", "Mobile"))
        self.table.verticalHeader().setVisible(False)  # Hide row numbers
        self.setCentralWidget(self.table)

    def load_data(self):
        """Load student data from database and display in table"""
        connection = sqlite3.connect("database.db")
        result = connection.execute("SELECT * FROM students")  # BUG: Should be "SELECT * FROM students"
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("register")
        button.clicked.connect(self.add_student)
        self.setLayout(layout)  # Layout set after all widgets added

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO student (name, course, mobile) VALUES (?, ?, ?)", \
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()  # BUG: 'main_window' not defined in this scope

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
sys.exit(app.exec())

add search student 

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout,QTableWidget, \
    QLabel, QWidget, QGridLayout, QLineEdit, QPushButton,QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox
from PyQt6.QtCore import Qt  # Import Qt for search flags
from PyQt6.QtGui import QAction
import sqlite3
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")
        edit_menu_item = self.menuBar().addMenu("&Edit")  # Added Edit menu

        add_student_action = QAction("Add Student", self)
        add_student_action.triggered.connect(self.insert)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)

        # Search action in Edit menu
        search_action = QAction("search", self)
        edit_menu_item.addAction(search_action)
        search_action.triggered.connect(self.search)

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Coures", "Mobile"))
        self.table.verticalHeader().setVisible(False)
        self.setCentralWidget(self.table)

    def load_data(self):
        """Load student data from database and display in table"""
        connection = sqlite3.connect("database.db")
        result = connection.execute("SELECT * FROM students")
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

    def search(self):  # Method to open search dialog
        dialog = SearchDialog()  # BUG: Should be SearchDialog
        dialog.exec()

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("register")
        button.clicked.connect(self.add_student)
        self.setLayout(layout)

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO student (name, course, mobile) VALUES (?, ?, ?)", \
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()

class SearchDialog(QDialog):  # Dialog class for searching students
    def __init__(self):
        super().__init__()
        #Set window title and size
        self.setWindowTitle("Search Student")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        # Create layout and input widget
        layout = QVBoxLayout()  # BUG: Should be self.layout or just layout
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("name")
        layout.addWidget(self.student_name)

        #Create button
        button = QPushButton("search")
        button.clicked.connect(self.search)
        layout.addWidget(button)

    def search(self):  # Search method
        name = self.student_name.text()
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        result = cursor.execute("SELECT * FROM student WHERE name = ?", (name,))  # BUG: Missing space in "SELECT *FROM"
        rows = list(result)
        print(rows)
        items = main_window.table.findItems(name, Qt.MatchFixedString)  # BUG: Should be Qt.MatchFixedString
        for item in items:
            print(item)
            main_window.table.item(item.row(), 1).setSelected(True)

        cursor.close()
        connection.close()

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
main_window.load_data()  # Load data when app starts
sys.exit(app.exec())

## Day 3



In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout, QTableWidget, \
    QStatusBar, QWidget, QGridLayout, QLineEdit, QPushButton, QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox, QToolBar
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QAction, QIcon
import sqlite3
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")
        self.setMinimumSize(800, 600)

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")
        edit_menu_item = self.menuBar().addMenu("&Edit")

        add_student_action = QAction(QIcon(r"icons\add.png"), "Add Student", self)
        add_student_action.triggered.connect(self.insert)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)

        search_action = QAction(QIcon("icons\search.png"), "Search", self)
        edit_menu_item.addAction(search_action)
        search_action.triggered.connect(self.search)

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Course", "Mobile"))
        self.table.verticalHeader().setVisible(False)
        self.setCentralWidget(self.table)

        #Create toolbar and add toolbar elements
        toolbar = QToolBar()
        toolbar.setMovable(True)
        self.addToolBar(toolbar)
        toolbar.addAction(add_student_action)
        toolbar.addAction(search_action)

        # Create status bar and add status bar elements
        self.statusbar = QStatusBar()
        self.setStatusBar(self.statusbar)

        # Detect a cell click
        self.table.cellClicked.connect(self.cell_clicked)

    def cell_clicked(self):
        edit_button = QPushButton("edit record")
        edit_button.clicked.connect(self.edit)

        delete_button = QPushButton("Delete record")
        delete_button.clicked.connect(self.delete)

        children = self.findChildren(QPushButton)
        if children:
            for child in children:
                self.statusbar.removeWidget(child)

        self.statusbar.addWidget(edit_button)
        self.statusbar.addWidget(delete_button)

    def load_data(self):
        """Load student data from database and display in table"""
        connection = sqlite3.connect("database.db")
        result = connection.execute("SELECT * FROM students")
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

    def search(self):
        dialog = SearchDialog()
        dialog.exec()

    def edit(self):
        dialog = EditDialog()
        dialog.exec()

    def delete(self):
        dialog = deleteDialog()
        dialog.exec()

class EditDialog(QDialog):
    pass

class deleteDialog(QDialog):
    pass

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Register")
        button.clicked.connect(self.add_student)
        layout.addWidget(button)
        self.setLayout(layout)

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        
        if not name or not mobile:
            print("Please fill all fields!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO students (name, course, mobile) VALUES (?, ?, ?)",
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()
        self.accept()


class SearchDialog(QDialog):
    def __init__(self):
        super().__init__()
        #Set window title and size
        self.setWindowTitle("Search Student")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        # Create layout and input widget
        layout = QVBoxLayout()
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        #Create button
        button = QPushButton("Search")
        button.clicked.connect(self.search_student)
        layout.addWidget(button)
        
        self.setLayout(layout)

    def search_student(self):
        name = self.student_name.text()
        if not name:
            print("Please enter a name to search!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        result = cursor.execute("SELECT * FROM students WHERE name = ?", (name,))
        rows = list(result)
        print(rows)
        
        # Find and select the row in the table
        items = main_window.table.findItems(name, Qt.MatchFlag.MatchFixedString)
        for item in items:
            print(item)
            main_window.table.item(item.row(), 0).setSelected(True)

        cursor.close()
        connection.close()
        self.accept()

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
main_window.load_data()
sys.exit(app.exec())

## Day 4

edit and delate button

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout, QTableWidget, \
    QStatusBar, QWidget, QGridLayout, QLineEdit, QPushButton, QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox, QToolBar  # Added QStatusBar, QToolBar
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QAction, QIcon  # Added QIcon for icons
import sqlite3
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")
        self.setMinimumSize(800, 600)  # Set minimum window size

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")
        edit_menu_item = self.menuBar().addMenu("&Edit")

        add_student_action = QAction(QIcon(r"icons\add.png"), "Add Student", self)  # Added icon
        add_student_action.triggered.connect(self.insert)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)

        search_action = QAction(QIcon("icons\search.png"), "Search", self)  # Added icon
        edit_menu_item.addAction(search_action)
        search_action.triggered.connect(self.search)

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Course", "Mobile"))  # Fixed spelling
        self.table.verticalHeader().setVisible(False)
        self.setCentralWidget(self.table)

        #Create toolbar and add toolbar elements
        toolbar = QToolBar()  # Create toolbar
        toolbar.setMovable(True)  # Allow toolbar to be moved
        self.addToolBar(toolbar)  # Add toolbar to window
        toolbar.addAction(add_student_action)  # Add actions to toolbar
        toolbar.addAction(search_action)

        # Create status bar and add status bar elements
        self.statusbar = QStatusBar()  # Create status bar
        self.setStatusBar(self.statusbar)  # Set status bar in window

        # Detect a cell click
        self.table.cellClicked.connect(self.cell_clicked)  # Connect cell click event

    def cell_clicked(self):  # Handle cell click event
        edit_button = QPushButton("edit record")  # Create edit button
        edit_button.clicked.connect(self.edit)  # Connect to edit method

        delete_button = QPushButton("Delete record")  # Create delete button
        delete_button.clicked.connect(self.delete)  # Connect to delete method

        children = self.findChildren(QPushButton)  # Find all existing buttons in status bar
        if children:
            for child in children:
                self.statusbar.removeWidget(child)  # Remove old buttons from status bar

        self.statusbar.addWidget(edit_button)  # Add edit button to status bar
        self.statusbar.addWidget(delete_button)  # Add delete button to status bar

    def load_data(self):
        """Load student data from database and display in table"""
        connection = sqlite3.connect("database.db")
        result = connection.execute("SELECT * FROM students")
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

    def search(self):
        dialog = SearchDialog()
        dialog.exec()

    def edit(self):  # Open edit dialog
        dialog = EditDialog()
        dialog.exec()

    def delete(self):  # Open delete dialog
        dialog = DeleteDialog()  # BUG: Should be DeleteDialog (not deleteDialog)
        dialog.exec()

class EditDialog(QDialog):  # NEW: Dialog for editing student records
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Update Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        # Get student name from selected row
        index = main_window.table.currentRow()  # Get selected row index
        student_name = main_window.table.item(index, 1).text()

        #Add student name widget
        self.student_name = QLineEdit(student_name)  # Pre-fill with existing name
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Get id from selected row
        self.student_id = main_window.table.item(index, 0).text()  # Store ID for update
        # Add combo box of courses
        course_name = main_window.table.item(index, 2).text()  # Get current course
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        self.course_name.setCurrentText(course_name)  # BUG: Should use setCurrentText (not setCurrentIndex)
        layout.addWidget(self.course_name)

        #Add mobile widget
        mobile = main_window.table.item(index, 3).text()
        self.mobile = QLineEdit(mobile)  # Pre-fill with existing mobile
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Update")
        button.clicked.connect(self.update_student)
        layout.addWidget(button)
        self.setLayout(layout)
    
    def update_student(self):  # Update student record in database
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("UPDATE students SET name = ?, course = ?, mobile = ? WHERE id = ?",  # BUG: Remove extra comma after mobile
                    (self.student_name.text(), self.course_name.itemText(self.course_name.currentIndex()), 
                    self.mobile.text(), self.student_id))
        
        connection.commit()
        cursor.close()
        connection.close()

        main_window.load_data()  # Refresh the table
        self.accept()  # Close dialog

class DeleteDialog(QDialog):  # NEW: Placeholder for delete dialog
    pass

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Register")
        button.clicked.connect(self.add_student)
        layout.addWidget(button)
        self.setLayout(layout)

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        
        if not name or not mobile:  # Validate input
            print("Please fill all fields!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO students (name, course, mobile) VALUES (?, ?, ?)",
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()
        self.accept()  # Close dialog after successful insertion

class SearchDialog(QDialog):
    def __init__(self):
        super().__init__()
        #Set window title and size
        self.setWindowTitle("Search Student")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        # Create layout and input widget
        layout = QVBoxLayout()
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        #Create button
        button = QPushButton("Search")
        button.clicked.connect(self.search_student)
        layout.addWidget(button)
        
        self.setLayout(layout)

    def search_student(self):
        name = self.student_name.text()
        if not name:  # Validate input
            print("Please enter a name to search!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        result = cursor.execute("SELECT * FROM students WHERE name = ?", (name,))
        rows = list(result)
        print(rows)
        
        # Find and select the row in the table
        items = main_window.table.findItems(name, Qt.MatchFlag.MatchFixedString)  # Correct Qt flag
        for item in items:
            print(item)
            main_window.table.item(item.row(), 0).setSelected(True)  # Select entire row

        cursor.close()
        connection.close()
        self.accept()  # Close dialog

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
main_window.load_data()
sys.exit(app.exec())

edit code

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout, QTableWidget, \
    QStatusBar, QWidget, QGridLayout, QLineEdit, QPushButton, QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox, QToolBar
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QAction, QIcon
import sqlite3
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")
        self.setMinimumSize(800, 600)

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")
        edit_menu_item = self.menuBar().addMenu("&Edit")

        add_student_action = QAction(QIcon(r"icons\add.png"), "Add Student", self)
        add_student_action.triggered.connect(self.insert)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)

        search_action = QAction(QIcon("icons\search.png"), "Search", self)
        edit_menu_item.addAction(search_action)
        search_action.triggered.connect(self.search)

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Course", "Mobile"))
        self.table.verticalHeader().setVisible(False)
        self.setCentralWidget(self.table)

        #Create toolbar and add toolbar elements
        toolbar = QToolBar()
        toolbar.setMovable(True)
        self.addToolBar(toolbar)
        toolbar.addAction(add_student_action)
        toolbar.addAction(search_action)

        # Create status bar and add status bar elements
        self.statusbar = QStatusBar()
        self.setStatusBar(self.statusbar)

        # Detect a cell click
        self.table.cellClicked.connect(self.cell_clicked)

    def cell_clicked(self):
        edit_button = QPushButton("edit record")
        edit_button.clicked.connect(self.edit)

        delete_button = QPushButton("Delete record")
        delete_button.clicked.connect(self.delete)

        children = self.findChildren(QPushButton)
        if children:
            for child in children:
                self.statusbar.removeWidget(child)

        self.statusbar.addWidget(edit_button)
        self.statusbar.addWidget(delete_button)

    def load_data(self):
        """Load student data from database and display in table"""
        connection = sqlite3.connect("database.db")
        result = connection.execute("SELECT * FROM students")
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

    def search(self):
        dialog = SearchDialog()
        dialog.exec()

    def edit(self):
        dialog = EditDialog()
        dialog.exec()

    def delete(self):
        dialog = DeleteDialog()
        dialog.exec()

class EditDialog(QDialog):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Update Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        # Get student name from selected row
        index = main_window.table.currentRow()
        student_name = main_window.table.item(index, 1).text()

        #Add student name widget
        self.student_name = QLineEdit(student_name)
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Get id from selected row
        self.student_id = main_window.table.item(index, 0).text()
        # Add combo box of courses
        course_name = main_window.table.item(index, 2).text()
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        
        # Find and set current course in combo box
        course_index = self.course_name.findText(course_name)
        if course_index >= 0:
            self.course_name.setCurrentIndex(course_index)
        layout.addWidget(self.course_name)

        #Add mobile widget
        mobile = main_window.table.item(index, 3).text()
        self.mobile = QLineEdit(mobile)
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Update")
        button.clicked.connect(self.update_student)
        layout.addWidget(button)
        self.setLayout(layout)
    
    def update_student(self):
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("UPDATE students SET name = ?, course = ?, mobile = ? WHERE id = ?", 
                    (self.student_name.text(), self.course_name.itemText(self.course_name.currentIndex()), 
                    self.mobile.text(), self.student_id))
        
        connection.commit()
        cursor.close()
        connection.close()

        main_window.load_data()  # Refresh the table

class DeleteDialog(QDialog):  # Placeholder for delete dialog
    pass

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Register")
        button.clicked.connect(self.add_student)
        layout.addWidget(button)
        self.setLayout(layout)

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        
        if not name or not mobile:  # Validate input
            print("Please fill all fields!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO students (name, course, mobile) VALUES (?, ?, ?)",
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()
        self.accept()  # Close dialog

class SearchDialog(QDialog):
    def __init__(self):
        super().__init__()
        #Set window title and size
        self.setWindowTitle("Search Student")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        # Create layout and input widget
        layout = QVBoxLayout()
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        #Create button
        button = QPushButton("Search")
        button.clicked.connect(self.search_student)
        layout.addWidget(button)
        
        self.setLayout(layout)

    def search_student(self):
        name = self.student_name.text()
        if not name:  # Validate input
            print("Please enter a name to search!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        result = cursor.execute("SELECT * FROM students WHERE name = ?", (name,))
        rows = list(result)
        print(rows)
        
        # Find and select the row in the table
        items = main_window.table.findItems(name, Qt.MatchFlag.MatchFixedString)
        for item in items:
            print(item)
            main_window.table.item(item.row(), 0).setSelected(True)

        cursor.close()
        connection.close()
        self.accept()  # Close dialog

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
main_window.load_data()
sys.exit(app.exec())

delete button

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout, QTableWidget, \
    QStatusBar, QWidget, QGridLayout, QLineEdit, QPushButton, QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox, QToolBar, QMessageBox, QLabel  # Added QMessageBox, QLabel
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QAction, QIcon
import sqlite3
import sys

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")
        self.setMinimumSize(800, 600)

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")
        edit_menu_item = self.menuBar().addMenu("&Edit")

        add_student_action = QAction(QIcon(r"icons\add.png"), "Add Student", self)
        add_student_action.triggered.connect(self.insert)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)

        search_action = QAction(QIcon("icons\search.png"), "Search", self)
        edit_menu_item.addAction(search_action)
        search_action.triggered.connect(self.search)

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Course", "Mobile"))
        self.table.verticalHeader().setVisible(False)
        self.setCentralWidget(self.table)

        #Create toolbar and add toolbar elements
        toolbar = QToolBar()
        toolbar.setMovable(True)
        self.addToolBar(toolbar)
        toolbar.addAction(add_student_action)
        toolbar.addAction(search_action)

        # Create status bar and add status bar elements
        self.statusbar = QStatusBar()
        self.setStatusBar(self.statusbar)

        # Detect a cell click
        self.table.cellClicked.connect(self.cell_clicked)

    def cell_clicked(self):
        edit_button = QPushButton("edit record")
        edit_button.clicked.connect(self.edit)

        delete_button = QPushButton("Delete record")
        delete_button.clicked.connect(self.delete)

        children = self.findChildren(QPushButton)
        if children:
            for child in children:
                self.statusbar.removeWidget(child)

        self.statusbar.addWidget(edit_button)
        self.statusbar.addWidget(delete_button)

    def load_data(self):
        """Load student data from database and display in table"""
        connection = sqlite3.connect("database.db")
        result = connection.execute("SELECT * FROM students")
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

    def search(self):
        dialog = SearchDialog()
        dialog.exec()

    def edit(self):
        dialog = EditDialog()
        dialog.exec()

    def delete(self):
        dialog = DeleteDialog()
        dialog.exec()

class EditDialog(QDialog):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Update Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        # Get student name from selected row
        index = main_window.table.currentRow()
        student_name = main_window.table.item(index, 1).text()

        #Add student name widget
        self.student_name = QLineEdit(student_name)
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Get id from selected row
        self.student_id = main_window.table.item(index, 0).text()
        # Add combo box of courses
        course_name = main_window.table.item(index, 2).text()
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
    
        course_index = self.course_name.findText(course_name)
        if course_index >= 0:
            self.course_name.setCurrentIndex(course_index)
        layout.addWidget(self.course_name)

        #Add mobile widget
        mobile = main_window.table.item(index, 3).text()
        self.mobile = QLineEdit(mobile)
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Update")
        button.clicked.connect(self.update_student)
        layout.addWidget(button)
        self.setLayout(layout)
    
    def update_student(self):
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("UPDATE students SET name = ?, course = ?, mobile = ? WHERE id = ?", 
                    (self.student_name.text(), self.course_name.itemText(self.course_name.currentIndex()), 
                    self.mobile.text(), self.student_id))
        
        connection.commit()
        cursor.close()
        connection.close()

        main_window.load_data()  # Refresh the table

class DeleteDialog(QDialog):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Delete Student Data")

        layout = QGridLayout()
        confirmation = QLabel("Are you sure want to delete?")  # Confirmation label
        yes = QPushButton("YES")
        no = QPushButton("NO")

        layout.addWidget(confirmation, 0, 0, 1, 2)
        layout.addWidget(yes, 1, 0)
        layout.addWidget(no, 1, 1)
        self.setLayout(layout)

        yes.clicked.connect(self.delete_student)
        no.clicked.connect(self.close)  # BUG: Missing connection for NO button

    def delete_student(self):
        index = main_window.table.currentRow()
        student_id = main_window.table.item(index, 0).text()

        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("DELETE FROM students WHERE id = ?", (student_id,))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()

        self.close()  # Close dialog

        QMessageBox.information(self, "Success", "The record was deleted successfully!")  # Show success message

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Register")
        button.clicked.connect(self.add_student)
        layout.addWidget(button)
        self.setLayout(layout)

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        
        if not name or not mobile:
            print("Please fill all fields!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO students (name, course, mobile) VALUES (?, ?, ?)",
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()
        self.accept()  # Close dialog

class SearchDialog(QDialog):
    def __init__(self):
        super().__init__()
        #Set window title and size
        self.setWindowTitle("Search Student")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        # Create layout and input widget
        layout = QVBoxLayout()
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        #Create button
        button = QPushButton("Search")
        button.clicked.connect(self.search_student)
        layout.addWidget(button)
        
        self.setLayout(layout)

    def search_student(self):
        name = self.student_name.text()
        if not name:
            print("Please enter a name to search!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        result = cursor.execute("SELECT * FROM students WHERE name = ?", (name,))
        rows = list(result)
        print(rows)
        
        # Find and select the row in the table
        items = main_window.table.findItems(name, Qt.MatchFlag.MatchFixedString)
        for item in items:
            print(item)
            main_window.table.item(item.row(), 0).setSelected(True)

        cursor.close()
        connection.close()
        self.accept()  # Close dialog

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
main_window.load_data()
sys.exit(app.exec())

## Day 5

create an about dialog

In [ ]:
from PyQt6.QtWidgets import QApplication, QVBoxLayout, QTableWidget, \
    QStatusBar, QWidget, QGridLayout, QLineEdit, QPushButton, QMainWindow, \
    QTableWidgetItem, QDialog, QComboBox, QToolBar, QMessageBox, QLabel
from PyQt6.QtCore import Qt
from PyQt6.QtGui import QAction, QIcon
import sqlite3
import sys

class Database_con:  # NEW: Database connection handler class
    def __init__(self, database_file="database.db"):
        self.database_file = database_file

    def connect(self):  # Establish and return database connection
        connection = sqlite3.connect(self.database_file)
        return connection

class MainWindow(QMainWindow):
    """Main window class for Student Management System with table and menu"""
    def __init__(self):
        """Initialize window, menu bar, and table widget"""
        super().__init__()
        self.setWindowTitle("Student Management System")
        self.setMinimumSize(800, 600)

        file_menu_item = self.menuBar().addMenu("&File")
        help_menu_item = self.menuBar().addMenu("&Help")
        edit_menu_item = self.menuBar().addMenu("&Edit")

        add_student_action = QAction(QIcon(r"icons\add.png"), "Add Student", self)
        add_student_action.triggered.connect(self.insert)
        file_menu_item.addAction(add_student_action)

        about_action = QAction("About", self)
        help_menu_item.addAction(about_action)
        about_action.setMenuRole(QAction.MenuRole.NoRole)
        about_action.triggered.connect(self.about)  # Connect About action to about method

        search_action = QAction(QIcon("icons\search.png"), "Search", self)
        edit_menu_item.addAction(search_action)
        search_action.triggered.connect(self.search)

        #Table widget setup
        self.table = QTableWidget()
        self.table.setColumnCount(4)
        self.table.setHorizontalHeaderLabels(("ID", "Name", "Course", "Mobile"))
        self.table.verticalHeader().setVisible(False)
        self.setCentralWidget(self.table)

        #Create toolbar and add toolbar elements
        toolbar = QToolBar()
        toolbar.setMovable(True)
        self.addToolBar(toolbar)
        toolbar.addAction(add_student_action)
        toolbar.addAction(search_action)

        # Create status bar and add status bar elements
        self.statusbar = QStatusBar()
        self.setStatusBar(self.statusbar)

        # Detect a cell click
        self.table.cellClicked.connect(self.cell_clicked)

    def cell_clicked(self):
        edit_button = QPushButton("edit record")
        edit_button.clicked.connect(self.edit)

        delete_button = QPushButton("Delete record")
        delete_button.clicked.connect(self.delete)

        children = self.findChildren(QPushButton)
        if children:
            for child in children:
                self.statusbar.removeWidget(child)

        self.statusbar.addWidget(edit_button)
        self.statusbar.addWidget(delete_button)

    def load_data(self):
        """Load student data from database and display in table"""
        connection = Database_con().connect()  # Using new database class
        result = connection.execute("SELECT * FROM students")
        self.table.setRowCount(0)
        for row_number, row_data in enumerate(result):
            self.table.insertRow(row_number)
            for column_number, data in enumerate(row_data):
                self.table.setItem(row_number, column_number, QTableWidgetItem(str(data)))
        connection.close()

    def insert(self):
        """Open dialog for inserting new student"""
        dialog = InsertDialog()
        dialog.exec()

    def search(self):
        dialog = SearchDialog()
        dialog.exec()

    def edit(self):
        dialog = EditDialog()
        dialog.exec()

    def delete(self):
        dialog = DeleteDialog()
        dialog.exec()

    def about(self):  # NEW: Method to open About dialog
        dialog = AboutDialog()
        dialog.exec()

class AboutDialog(QMessageBox):  # NEW: About dialog class
    def __init__(self):
        super().__init__()
        self.setWindowTitle("About")
        content = """
        This app was created during the "alimrk"
        feel free to modify and reuse this app.
        """
        self.setText(content)

class EditDialog(QDialog):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Update Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        # Get student name from selected row
        index = main_window.table.currentRow()
        student_name = main_window.table.item(index, 1).text()

        #Add student name widget
        self.student_name = QLineEdit(student_name)
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Get id from selected row
        self.student_id = main_window.table.item(index, 0).text()
        # Add combo box of courses
        course_name = main_window.table.item(index, 2).text()
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
    
        course_index = self.course_name.findText(course_name)
        if course_index >= 0:
            self.course_name.setCurrentIndex(course_index)
        layout.addWidget(self.course_name)

        #Add mobile widget
        mobile = main_window.table.item(index, 3).text()
        self.mobile = QLineEdit(mobile)
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Update")
        button.clicked.connect(self.update_student)
        layout.addWidget(button)
        self.setLayout(layout)
    
    def update_student(self):
        connection = Database_con().connect()  # Using new database class
        cursor = connection.cursor()
        cursor.execute("UPDATE students SET name = ?, course = ?, mobile = ? WHERE id = ?", 
                    (self.student_name.text(), self.course_name.itemText(self.course_name.currentIndex()), 
                    self.mobile.text(), self.student_id))
        
        connection.commit()
        cursor.close()
        connection.close()

        main_window.load_data()  # Refresh the table

class DeleteDialog(QDialog):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Delete Student Data")

        layout = QGridLayout()
        confirmation = QLabel("Are you sure want to delete?")
        yes = QPushButton("YES")
        no = QPushButton("NO")

        layout.addWidget(confirmation, 0, 0, 1, 2)
        layout.addWidget(yes, 1, 0)
        layout.addWidget(no, 1, 1)
        self.setLayout(layout)

        yes.clicked.connect(self.delete_student)
        no.clicked.connect(self.close)  # BUG: Missing connection for NO button

    def delete_student(self):
        index = main_window.table.currentRow()
        student_id = main_window.table.item(index, 0).text()

        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("DELETE FROM students WHERE id = ?", (student_id,))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()

        self.close()

        QMessageBox.information(self, "Success", "The record was deleted successfully!")

class InsertDialog(QDialog):
    """Dialog for adding new student records to database"""
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insert Student Data")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        layout = QVBoxLayout()

        #Add student name widget
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        # Add combo box of courses
        self.course_name = QComboBox()
        courses = ["Biology", "Math", "Astronomy", "Physics"]
        self.course_name.addItems(courses)
        layout.addWidget(self.course_name)

        #Add mobile widget
        self.mobile = QLineEdit()
        self.mobile.setPlaceholderText("Mobile")
        layout.addWidget(self.mobile)

        # Add a submit button
        button = QPushButton("Register")
        button.clicked.connect(self.add_student)
        layout.addWidget(button)
        self.setLayout(layout)

    def add_student(self):
        """Insert new student into database and refresh main table"""
        name = self.student_name.text()
        course = self.course_name.itemText(self.course_name.currentIndex())
        mobile = self.mobile.text()
        
        if not name or not mobile:
            print("Please fill all fields!")
            return
            
        connection = sqlite3.connect("database.db")
        cursor = connection.cursor()
        cursor.execute("INSERT INTO students (name, course, mobile) VALUES (?, ?, ?)",
                        (name, course, mobile))
        connection.commit()
        cursor.close()
        connection.close()
        main_window.load_data()
        self.accept()

class SearchDialog(QDialog):
    def __init__(self):
        super().__init__()
        #Set window title and size
        self.setWindowTitle("Search Student")
        self.setFixedWidth(300)
        self.setFixedHeight(300)

        # Create layout and input widget
        layout = QVBoxLayout()
        self.student_name = QLineEdit()
        self.student_name.setPlaceholderText("Name")
        layout.addWidget(self.student_name)

        #Create button
        button = QPushButton("Search")
        button.clicked.connect(self.search_student)
        layout.addWidget(button)
        
        self.setLayout(layout)

    def search_student(self):
        name = self.student_name.text()
        if not name:
            print("Please enter a name to search!")
            return
            
        connection = Database_con().connect()  # Using new database class
        cursor = connection.cursor()
        result = cursor.execute("SELECT * FROM students WHERE name = ?", (name,))
        rows = list(result)
        print(rows)
        
        # Find and select the row in the table
        items = main_window.table.findItems(name, Qt.MatchFlag.MatchFixedString)
        for item in items:
            print(item)
            main_window.table.item(item.row(), 0).setSelected(True)

        cursor.close()
        connection.close()
        self.accept()

app = QApplication(sys.argv)
main_window = MainWindow()
main_window.show()
main_window.load_data()
sys.exit(app.exec())